# Step 4: Performance Evaluation and Explainable AI (SHAP)

## Overview
Evaluates the predictive performance of trained Random Forest, XGBoost, and LightGBM models on the test set.

### Evaluation Metrics
- **ROC-AUC & PR-AUC**: Discriminating capacity across probability thresholds.
- **Classification Performance**: Accuracy, Sensitivity (Recall), Specificity, Precision, and F1-score.
- **Confusion Matrix**: True Positive, False Positive, True Negative, and False Negative counts.

### Model Interpretation
- **SHAP (SHapley Additive exPlanations)** values computed via TreeExplainer to explain individual risk scores and capture global feature importances.

## Inputs and Outputs
- **Input**: Trained model files and test partition data
- **Output**: ROC curves, PR curves, SHAP value summary plots, confusion matrices



In [ ]:
import shap
import pandas as pd
import numpy as np 
import os
import joblib
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import xgboost as xgb
import seaborn as sns

In [ ]:
data_dir = '../../data'
input_dir = f'{data_dir}/step3_model_outputs' 
output_dir = f'{data_dir}/step4_evaluation_outputs'

file_paths = {
    'X_test_scaled_df': f'{input_dir}/step3_all_x_test_scaled',
    'y_test_df': f'{input_dir}/step3_all_y_test',
    'X_train_scaled_df': f'{input_dir}/step3_all_x_train_scaled',
    'y_train_df': f'{input_dir}/step3_all_y_train',
    'tagt_feat': f'{data_dir}/step2_processing_outputs/step2_all_tagt_feat'
}

In [ ]:
# Dict to store loaded DataFrames
dataframes = {}

# 辞書の各ファイルパスに対してpd.read_paqcetを実行
for key, file_path in file_paths.items():
    if not os.path.exists(file_path):
        print(f"Warning: ファイル '{file_path}' が存在しません。スキップします。")
        continue
        
    try:
        # lineterminator='\n' を指定
        df = pd.read_parquet(file_path)
        dataframes[key] = df
        print(f"'{key}'loaded successfully. Shape: {df.shape}")
    except Exception as e:
        print(f"Error: '{file_path}' failed to load. Error: {e}")

In [ ]:
# Load pre-trained models
rf_model = joblib.load( f'{input_dir}/all_rf_model.joblib')
xgb_model = joblib.load( f'{input_dir}/all_xgb_model.joblib')
lgb_model = joblib.load( f'{input_dir}/all_lgb_model.joblib')

In [ ]:
# Load scaled data partitions
X_test_scaled_df = dataframes['X_test_scaled_df']
X_train_scaled_df = dataframes['X_train_scaled_df']
y_test = dataframes['y_test_df']
y_train = dataframes['y_train_df']

In [ ]:
# モデルの評価 
rf_y_pred = rf_model.predict(X_test_scaled_df)
xgb_y_pred = xgb_model.predict(X_test_scaled_df)
lgb_y_pred = lgb_model.predict(X_test_scaled_df)

print("--- モデル評価 RF ---")
# 精度 (Accuracy)
print(f"Accuracy: {accuracy_score(y_test, rf_y_pred):.4f}")

print("--- モデル評価 XGB ---")
# 精度 (Accuracy)
print(f"Accuracy: {accuracy_score(y_test, xgb_y_pred):.4f}")

print("--- モデル評価 LightGBM ---")
# 精度 (Accuracy)
print(f"Accuracy: {accuracy_score(y_test, lgb_y_pred):.4f}")

# 詳細な評価レポート (適合率、再現率、F1スコア)
print("\nClassification Report(random forest):")
print(classification_report(y_test, rf_y_pred))
print("\nClassification Report(xg boost):")
print(classification_report(y_test, xgb_y_pred))
print("\nClassification Report(light gbm):")
print(classification_report(y_test, lgb_y_pred))

In [ ]:
#AUCの計算
rf_y_probs = rf_model.predict_proba(X_test_scaled_df)[:, 1]

auc_score = roc_auc_score(y_test, rf_y_probs)
print(f"ROC-AUC Score: {auc_score:.4f}")

fpr, tpr, thresholds = roc_curve(y_test, rf_y_probs)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'RF (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], 'k--') 
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve(Random Forest)')
plt.legend()
plt.show()

In [ ]:
rf_y_probs_df = pd.DataFrame(rf_y_probs)
rf_y_probs_df.columns = xgb_y_probs_df.columns.astype(str)
rf_y_probs_df.to_parquet(f'{data_dir}/step4_evaluation_outputs/rf_y_probs')

In [ ]:
#AUCの計算
xgb_y_probs = xgb_model.predict_proba(X_test_scaled_df)[:, 1]

auc_score = roc_auc_score(y_test, xgb_y_probs)
print(f"ROC-AUC Score: {auc_score:.4f}")

fpr, tpr, thresholds = roc_curve(y_test, xgb_y_probs)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'XGB (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], 'k--') 
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve(XGBoost)')
plt.legend()
plt.show()

In [ ]:
xgb_y_probs_df = pd.DataFrame(xgb_y_probs)
xgb_y_probs_df.columns = xgb_y_probs_df.columns.astype(str)
xgb_y_probs_df.to_parquet(f'{data_dir}/step4_evaluation_outputs/xgb_y_probs')

In [ ]:
lgb_y_probs = lgb_model.predict_proba(X_test_scaled_df)[:, 1]
print(f"ROC-AUC Score: {roc_auc_score(y_test, lgb_y_probs):.4f}")

fpr, tpr, thresholds = roc_curve(y_test, lgb_y_probs)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'LGB (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], 'k--') # 対角線
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve(LightGBM)')
plt.legend()
plt.show()

In [ ]:
lgb_y_probs_df = pd.DataFrame(lgb_y_probs)
lgb_y_probs_df.columns = xgb_y_probs_df.columns.astype(str)
lgb_y_probs_df.to_parquet(f'{data_dir}/step4_evaluation_outputs/lgb_y_probs')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import shap

# フォント設定
mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['font.size'] = 18

# 1. SHAP値計算
model = lgb_model
explainer = shap.TreeExplainer(model.booster_)
shap_values = explainer(X_test_scaled_df)

# 2. 名前の置換
name_map = {
    "4291427": "Nivolumab (ID: 4291427)", 
    "4240403": "Etoposide (ID: 4240403)", 
    "DIAGNOSIS_C50": "Breast Cancer (C50)",
    "lastWBC": "Latest White Blood Cell Count",
    "DIAGNOSIS_C77": "Secondary Malignant Neoplasm of Lymph Nodes (C77)",
    "lastNEUT#": "Latest Absolute Neutrophil Count",
    "BPH": "Systolic Blood Pressure",
    "4291435": "Pembrolizumab (ID: 4291435)",
    "MS_HEIGHT":"HEIGHT"
}
    
shap_values.feature_names = [name_map.get(n, n) for n in shap_values.feature_names]

# 3. 描画
plt.figure(figsize=(10, 6))
# LightGBMの正例(1)を選択
shap.plots.beeswarm(shap_values[:, :, 1], max_display=11, show=False)

save_path = os.path.join(output_dir, 'Fig3_final_SHAP_Plot.tiff')

# 軸ラベルの最終調整
ax = plt.gca()

# 1. X軸のラベルのフォントサイズを設定
ax.set_xlabel('SHAP value', fontsize=18, fontweight='bold')

# 2. Y軸のフォントサイズを18に設定
ax.tick_params(axis='y', labelsize=18)

# 3. X軸のフォントサイズを18に設定
ax.tick_params(axis='x', labelsize=18)

# 4. カラーバーのフォントサイズを18に設定
for extra_ax in plt.gcf().axes:
    extra_ax.tick_params(labelsize=18)
    for text in extra_ax.get_images():
        pass
    
plt.tight_layout()
plt.savefig(save_path, dpi=300)
plt.show()

In [ ]:
# 必要なモジュールの追加インポート
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss
import matplotlib.pyplot as plt

# 予測確率の辞書を作成（既存の変数を活用）
model_probs = {
    'RF': rf_y_probs,
    'XGB': xgb_y_probs,
    'LightGBM': lgb_y_probs
}

# 描画のセットアップ (上部にキャリブレーションカーブ、下部に予測確率のヒストグラム)
plt.figure(figsize=(10, 10))
ax1 = plt.subplot2grid((3, 1), (0, 0), rowspan=2)
ax2 = plt.subplot2grid((3, 1), (2, 0))

# 対角線
ax1.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")

for name, probs in model_probs.items():
    # 1. ブライアスコアとLog Lossの計算
    brier = brier_score_loss(y_test, probs)
    logloss = log_loss(y_test, probs)
    
    print(f"--- {name} ---")
    print(f"Brier Score : {brier:.4f}")
    print(f"Log Loss    : {logloss:.4f}\n")
    
    # 2. キャリブレーションカーブのデータ取得 (10分割)
    fraction_of_positives, mean_predicted_value = calibration_curve(y_test, probs, n_bins=10)
    
    # 上部: キャリブレーションカーブのプロット
    ax1.plot(mean_predicted_value, fraction_of_positives, "s-", 
             label=f"{name} (Brier={brier:.3f})")
    
    # 下部: 予測確率の分布ヒストグラム
    ax2.hist(probs, range=(0, 1), bins=10, label=name, histtype="step", lw=2)

# グラフの体裁を整える
ax1.set_ylabel("Fraction of positives (True Probability)")
ax1.set_title("Calibration Curves (Reliability Diagram)")
ax1.legend(loc="lower right")
ax1.grid(True, alpha=0.3)

ax2.set_xlabel("Mean predicted value (Predicted Probability)")
ax2.set_ylabel("Count")
ax2.legend(loc="upper center", ncol=3)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()